# Data Leakage Analysis (Startup Case)

This notebook demonstrates how to check for target and entity leakage between train and test splits, simulating a real-world startup scenario. The approach is based on guardrails for tabular ML pipelines.

In [ ]:
# Install dependencies if running in Colab
!pip install pandas numpy scikit-learn

## Leakage Detection Function

This function computes the intersection of entities, duplicate rows, and target drift between a training dataset and a test set.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

def evaluate_data_leakage(
    df_train: pd.DataFrame, df_test: pd.DataFrame, id_column=None, target_column=None
):
    """
    Run standard data leakage checks between training and test sets.
    """
    report = {}

    # 1. Row Leakage (Exact duplicates)
    # Remove the target to ensure we compare features only
    # (even if labels differ by mistake, identical features are still a problem)
    cols_to_check = [c for c in df_train.columns if c != target_column]

    # Deduplicate both sides before merging to avoid Cartesian products
    train_deduped = df_train[cols_to_check].drop_duplicates()
    test_deduped = df_test[cols_to_check].drop_duplicates()

    # Inner merge to find identical rows based on features
    intersection = pd.merge(train_deduped, test_deduped, how="inner")
    leaked_rows = len(intersection)

    report["Exact duplicate rows (Leakage)"] = leaked_rows
    report["% Leakage on Test Set"] = (leaked_rows / len(df_test)) * 100 if len(df_test) > 0 else 0

    # 2. Entity Leakage (ID overlap)
    if id_column and id_column in df_train.columns:
        id_train = set(df_train[id_column].dropna().unique())
        id_test = set(df_test[id_column].dropna().unique())

        shared_entities = id_train.intersection(id_test)
        report["Shared entities (IDs)"] = len(shared_entities)

    # 3. Target Drift (Statistical check)
    if target_column and target_column in df_train.columns:
        mean_train = df_train[target_column].mean()
        mean_test = df_test[target_column].mean()
        report["Target mean difference"] = abs(mean_train - mean_test)

    return report

## Simulate a Leaky Pipeline

Let's generate synthetic user data and intentionally introduce a leak by copying data across the train/test split boundary, then use our function to detect it.

In [ ]:
# 1. Create a synthetic dataset
X, y = make_classification(n_samples=5000, n_features=20, random_state=42)
df = pd.DataFrame(X, columns=[f"feat_{i}" for i in range(20)])
df["target"] = y
df["user_id"] = np.random.randint(1, 1000, size=5000)  # Simulate user IDs

# 2. Split (intentionally introduce some leakage to test the function)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

# INJECT DELIBERATE LEAKAGE: copy 50 rows from train to test
test_df = pd.concat([test_df, train_df.head(50)])

# 3. Evaluate Leakage
print("--- Pre-Training Leakage Analysis ---")
results = evaluate_data_leakage(
    train_df, test_df, id_column="user_id", target_column="target"
)

for metric, value in results.items():
    if isinstance(value, float):
        print(f"{metric}: {value:.2f}")
    else:
        print(f"{metric}: {value}")

# 4. Conclusions
if results["Exact duplicate rows (Leakage)"] == 0:
    print("\nNo leakage detected. Proceeding to training...")
else:
    print("\nWARNING: Data leakage detected! Clean the dataset before proceeding.")